In [2]:
import torch 

In [3]:
env_name = 'CartPole-v1'

In [4]:
import gymnasium as gym 

In [5]:
env = gym.make(env_name, render_mode = 'human')

In [6]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [7]:
env.action_space

Discrete(2)

In [8]:
obs_dim = env.observation_space.shape[0]

In [9]:
n_acts = env.action_space.n

### Lets make the NN which will learn our policy

In [10]:
import torch.nn as nn
hidden_sizes = [32]
layers = []
sizes = [obs_dim] + hidden_sizes + [n_acts]
activation = nn.Tanh
output_activation = nn.Identity
for j in range(len(sizes)-1):
    act = activation if j < len(sizes)-2 else output_activation
    layers += [nn.Linear(sizes[j], sizes[j+1]), act()]
mlp =  nn.Sequential(*layers)

In [11]:
logits_net = mlp

In [12]:
from torch.distributions.categorical import Categorical
def get_policy(obs):
    logits = logits_net(obs)
    return Categorical(logits=logits)

def get_action(obs):
    return get_policy(obs).sample().item()

def compute_loss(obs, act, weights):
    logp = get_policy(obs).log_prob(act)
    return -(logp * weights).mean()


In [13]:
from torch.optim import Adam
import numpy as np
lr = 1e-2
optimizer = Adam(logits_net.parameters(), lr=lr)

In [14]:
render = True
def train_one_epoch():
    batch_obs = []
    batch_acts = []
    batch_weights = []
    batch_rets = []
    batch_lens = []

    obs, _ = env.reset()
    done = False
    ep_rews = []

    finished_rendering_this_epoch = False
    while True:

        if (not finished_rendering_this_epoch) and render:
            env.render()
        batch_obs.append(obs.copy())

        act = get_action(torch.as_tensor(obs, dtype=torch.float32))
        obs, rew, done, _, _ = env.step(act)

        batch_acts.append(act)
        ep_rews.append(rew)

        if done:
            ep_ret, ep_len = sum(ep_rews), len(ep_rews)
            batch_rets.append(ep_ret)
            batch_lens.append(ep_len)

            batch_weights += [ep_ret] * ep_len
            (obs, _), done, ep_rews = env.reset(), False, []
            finished_rendering_this_epoch = True

            if len(batch_obs) > batch_size:
                break
    optimizer.zero_grad()
    batch_loss = compute_loss(obs=torch.as_tensor(batch_obs, dtype=torch.float32),
                              act=torch.as_tensor(batch_acts, dtype=torch.int32),
                              weights=torch.as_tensor(batch_weights, dtype=torch.float32)
                             )
    batch_loss.backward()
    optimizer.step()
    return batch_loss, batch_rets, batch_lens

        

In [ ]:
# training loop
epochs = 100
batch_size = 5000

for i in range(epochs):
    batch_loss, batch_rets, batch_lens = train_one_epoch()
    print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f' % 
        (i, batch_loss, np.mean(batch_rets), np.mean(batch_lens)))
    

/var/folders/1k/65njxlw90c12klrbvtxc0w_40000gn/T/ipykernel_87560/2478802725.py:38: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  batch_loss = compute_loss(obs=torch.as_tensor(batch_obs, dtype=torch.float32),
/var/folders/1k/65njxlw90c12klrbvtxc0w_40000gn/T/ipykernel_87560/3415372924.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:823.)
  print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f' %


epoch:   0 	 loss: 19.744 	 return: 22.217 	 ep_len: 22.217
